In [1]:
import os
import pydicom
import h5py
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn.functional as F

In [2]:
import os
import pydicom
import h5py
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


def make_dir(path, exist_ok=True):
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=exist_ok)
        os.chmod(path, 0o777)


def resize_volume_torch(volume_np, target_shape=(192, 192)):
    vol = torch.tensor(volume_np, dtype=torch.float32).unsqueeze(1)  # (Z, 1, H, W)
    vol_resized = F.interpolate(vol, size=target_shape, mode='bilinear', align_corners=False)
    return vol_resized.squeeze(1).numpy().astype(np.int16)  # (Z, H, W)


def get_bvalue(ds):
    """DWI b-value 추출. Siemens private tag(0019,100c)를 우선 쓰고,
    없으면 표준 DiffusionBValue 태그를 시도한다. 둘 다 없으면 -1(unknown)."""
    tag = ds.get((0x0019, 0x100c), None)
    if tag is not None:
        try:
            return int(str(tag.value).split('\\')[0])
        except (ValueError, TypeError):
            pass
    bval = getattr(ds, 'DiffusionBValue', None)
    if bval is not None:
        try:
            return int(bval)
        except (ValueError, TypeError):
            pass
    return -1


def load_dicom_volume(dicom_folder, target_shape=(192, 192), extra_info=None):
    all_files = sorted([f for f in os.listdir(dicom_folder) if f.endswith(".dcm")])
    slices, instance_numbers, file_paths, bvalues = [], [], [], []
    extra_data = {key: [] for key in extra_info} if extra_info else {}

    for f in all_files:
        path = os.path.join(dicom_folder, f)
        ds = pydicom.dcmread(path)
        slices.append(ds.pixel_array)
        instance_numbers.append(int(ds.InstanceNumber))
        file_paths.append(path)
        bvalues.append(get_bvalue(ds))

        if extra_info is not None:
            for key in extra_info:
                value = getattr(ds, key, "")
                if isinstance(value, pydicom.multival.MultiValue):
                    value = list(value)
                extra_data[key].append(value)

    sorted_indices = np.argsort(instance_numbers)
    slices = [slices[i] for i in sorted_indices]
    file_paths = [file_paths[i] for i in sorted_indices]
    bvalues = [bvalues[i] for i in sorted_indices]
    for key in extra_data:
        extra_data[key] = [extra_data[key][i] for i in sorted_indices]
    extra_data['b_value'] = bvalues

    volume = np.stack(slices)
    volume_resized = resize_volume_torch(volume, target_shape)
    return volume_resized, file_paths, extra_data


def resize_mask_volume_torch(volume_np, target_shape=(192, 192)):
    vol = torch.tensor(volume_np, dtype=torch.float32).unsqueeze(1)  # (Z, 1, H, W)
    vol_resized = F.interpolate(vol, size=target_shape, mode='nearest')
    return vol_resized.squeeze(1).numpy().astype(np.uint8)


def load_mask_volume(labelmask_folder, dcm_files, target_shape=(192, 192)):
    mask_volume = []
    for f in dcm_files:
        base = os.path.splitext(os.path.basename(f))[0]
        png_path = os.path.join(labelmask_folder, f"{base}.png")
        if os.path.exists(png_path):
            mask = Image.open(png_path).convert("L")
            mask_np = (np.array(mask) > 0).astype(np.uint8)
        else:
            mask_np = None
        mask_volume.append(mask_np)

    default_shape = next((m.shape for m in mask_volume if m is not None), target_shape)
    for i in range(len(mask_volume)):
        if mask_volume[i] is None:
            mask_volume[i] = np.zeros(default_shape, dtype=np.uint8)

    stacked = np.stack(mask_volume)
    return resize_mask_volume_torch(stacked, target_shape)


def resolve_case_dir(patient_dir, case):
    """환자마다 'pre'/'post' 또는 'preop'/'postop' 중 실제 존재하는 폴더명을 찾는다."""
    for name in (case, case + "op"):
        p = os.path.join(patient_dir, name)
        if os.path.isdir(p):
            return p
    return None


def get_pairs(patient_dir, extra_info=None):
    """pre/post를 각각 독립적으로 처리. 해당 case의 dicom 폴더 또는 labelmask 폴더가
    없으면 그 case만 건너뛴다 (라벨 없는 case를 '정상'으로 임의 추정하지 않음)."""
    results = {}
    for case in ['pre', 'post']:
        input_path = resolve_case_dir(patient_dir, case)
        label_path = os.path.join(patient_dir, "labelmask", case)

        if input_path is None:
            continue
        if not os.path.isdir(label_path):
            continue

        dcm_volume, dcm_files, extra_data = load_dicom_volume(input_path, extra_info=extra_info)
        mask_volume = load_mask_volume(label_path, dcm_files)
        results[case] = (dcm_files, dcm_volume, mask_volume, extra_data)

    return results


def get_h5(root_path, out_root=None, extra_info=None, limit_patients=None):
    out_root = out_root or os.path.join(root_path, 'h5')
    n_saved = 0
    n_skipped = 0
    n_patients = 0

    for type_id in sorted(os.listdir(root_path)):
        type_path = os.path.join(root_path, type_id)
        if not os.path.isdir(type_path) or type_id == os.path.basename(out_root):
            continue

        for patient_id in sorted(os.listdir(type_path)):
            patient_dir = os.path.join(type_path, patient_id)
            if not os.path.isdir(patient_dir):
                continue

            if limit_patients is not None and n_patients >= limit_patients:
                return n_saved, n_skipped

            n_patients += 1
            pairs = get_pairs(patient_dir, extra_info=extra_info)
            if not pairs:
                print(f'[skip] no usable pre/post + label pair: {patient_dir}')
                n_skipped += 1
                continue

            save_dir = os.path.join(out_root, type_id)
            make_dir(save_dir)

            for case, (dcm_files, dcm_volume, mask_volume, extra_data) in pairs.items():
                save_path = os.path.join(save_dir, f"{patient_id}_{case}.h5")
                with h5py.File(save_path, 'w') as f:
                    f.create_dataset('meta', data=dcm_files)
                    f.create_dataset('dcm', data=dcm_volume)
                    f.create_dataset('label', data=mask_volume)
                    f.create_dataset('b_value', data=np.array(extra_data['b_value']))
                    if extra_info:
                        for key in extra_info:
                            f.create_dataset(key, data=np.array(extra_data[key]))
                n_saved += 1
                print(f'[saved] {save_path}  slices={dcm_volume.shape[0]}  lesion_slices={(mask_volume.sum(axis=(1,2))>0).sum()}')

    return n_saved, n_skipped

## 실제 데이터 실행 (아래 셀들은 출력 미포함)

아래 셀들(`root_path`, `out_root` 지정 이하)은 원내(양산부산대학교병원) 실데이터 경로(`/workspace/data/...`)를 대상으로 실행하는 코드입니다. 이 셀들의 출력에는 실제 환자 폴더명·h5 파일 경로가 그대로 찍히기 때문에(예: `[saved] .../<환자번호>_<환자명>_pre.h5`, `f['meta'][0]`이 원본 DICOM 절대경로), **개인정보 유출 방지를 위해 셀 출력을 의도적으로 비워둔 채로** 커밋합니다.

이 파이프라인이 실제로 정상 동작한다는 실행 증거는 노트북 맨 아래 "실행 검증용 데모(합성 익명 데이터)" 섹션에서 대신 확인할 수 있습니다 — 동일한 `get_h5` 함수를 무작위 합성 DICOM에 대해 실행하고 그 결과를 출력합니다.

In [ ]:
root_path = '/workspace/data/results_train_cleaned'
out_root = '/workspace/data/h5'
n_saved, n_skipped = get_h5(root_path, out_root=out_root, extra_info=['SliceThickness'])
print(f'saved={n_saved} skipped_patients={n_skipped}')

In [ ]:
f = h5py.File(list(__import__('glob').glob(out_root + '/**/*.h5', recursive=True))[0])


In [ ]:
f.keys()

In [ ]:
f['meta'][0].decode('utf-8')

In [ ]:
import pydicom

def resize_img_torch(img_np, target_shape=(192, 192)):
    img = torch.tensor(img_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
    resized = F.interpolate(img, size=target_shape, mode='bilinear', align_corners=False)
    return resized.squeeze().numpy().astype(np.int16)  # (H, W)

def dcm2img(d):
    dcm = pydicom.dcmread(d)
    return resize_img_torch(dcm.pixel_array)


In [ ]:

plt.subplot(1,3,1)
plt.imshow(f['dcm'][0], cmap='gray')
plt.subplot(1,3,2)
plt.imshow(dcm2img(f['meta'][0].decode('utf-8')), cmap='gray')
plt.subplot(1,3,3)
plt.imshow(f['dcm'][0] - dcm2img(f['meta'][0].decode('utf-8')), cmap ='bwr')
plt.colorbar()

## 실행 검증용 데모 (합성 익명 데이터)

`get_h5`(위 `3a75b6b8` 셀에서 정의) 함수가 실제로 정상 동작함을 보이기 위해, **실제 환자 데이터가 아닌 무작위로 생성한 합성 DICOM+라벨마스크**로 최소 구성(환자 1명, pre 4슬라이스, 그중 1슬라이스에 합성 병변)을 임시 폴더에 만들고 동일한 `get_h5` 파이프라인을 실행합니다. 실행 후 h5 내용을 확인하고 임시 폴더는 삭제합니다.

In [3]:
import tempfile
import shutil
import glob
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid


def _make_synthetic_dicom(path, rows=64, cols=64, instance_number=1, b_value=1000, seed=0):
    """실행 검증용 합성 DICOM 1장 생성 (실제 환자 데이터 아님)."""
    rng = np.random.default_rng(seed)
    pixel_array = (rng.random((rows, cols)) * 200).astype(np.uint16)
    pixel_array[20:28, 20:28] += 400

    file_meta = FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = pydicom.uid.MRImageStorage
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    ds = FileDataset(path, {}, file_meta=file_meta, preamble=b"\0" * 128)
    ds.PatientName = "SYNTHETIC^DEMO"
    ds.PatientID = "SYNTH0000"
    ds.Modality = "MR"
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.InstanceNumber = instance_number
    ds.Rows, ds.Columns = rows, cols
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 0
    ds.PixelData = pixel_array.tobytes()
    ds.is_little_endian = True
    ds.is_implicit_VR = False
    ds.add_new(0x00190010, 'LO', 'SIEMENS MR HEADER')
    ds.add_new(0x0019100c, 'IS', str(b_value))
    ds.SliceThickness = 5.0

    os.makedirs(os.path.dirname(path), exist_ok=True)
    ds.save_as(path, write_like_original=False)
    return pixel_array


demo_root = tempfile.mkdtemp(prefix="dicom2h5_demo_")
try:
    demo_data_root = os.path.join(demo_root, "results_train_cleaned")
    demo_out_root = os.path.join(demo_root, "h5")

    # 완전 익명 케이스 ID(case_001)로 최소 구성(pre 4슬라이스 + labelmask) 생성
    patient_dir = os.path.join(demo_data_root, "demo_group", "case_001")
    pre_dir = os.path.join(patient_dir, "pre")
    label_dir = os.path.join(patient_dir, "labelmask", "pre")
    os.makedirs(label_dir, exist_ok=True)

    n_slices = 4
    for i in range(n_slices):
        fname = f"{i}-dicom-000{i}.dcm"
        pixel_array = _make_synthetic_dicom(
            os.path.join(pre_dir, fname), instance_number=i, b_value=1000, seed=i
        )
        mask = np.zeros_like(pixel_array, dtype=np.uint8)
        if i == 2:
            mask[20:28, 20:28] = 255  # 슬라이스 2에만 작은 합성 병변 마스크
        Image.fromarray(mask).save(os.path.join(label_dir, f"{i}-dicom-000{i}.png"))

    demo_n_saved, demo_n_skipped = get_h5(demo_data_root, out_root=demo_out_root, extra_info=['SliceThickness'])
    print(f"[DEMO] saved={demo_n_saved} skipped_patients={demo_n_skipped}")

    demo_h5_path = sorted(glob.glob(demo_out_root + '/**/*.h5', recursive=True))[0]
    with h5py.File(demo_h5_path) as demo_f:
        print(f"[DEMO] h5 keys: {list(demo_f.keys())}")
        print(f"[DEMO] dcm shape: {demo_f['dcm'].shape}, label shape: {demo_f['label'].shape}")
        print(f"[DEMO] b_value: {demo_f['b_value'][:]}")
        print(f"[DEMO] lesion voxel count per slice: {demo_f['label'][:].sum(axis=(1, 2))}")
        print(f"[DEMO] meta[0] (합성 익명 경로): {demo_f['meta'][0].decode('utf-8').replace(demo_root, '<demo_tmp>')}")

    assert demo_n_saved == 1 and demo_n_skipped == 0
    print("\n[DEMO] get_h5 파이프라인이 합성 데이터에서 정상 실행됨을 확인.")
finally:
    shutil.rmtree(demo_root, ignore_errors=True)


[saved] /tmp/dicom2h5_demo_x1fxfbjq/h5/demo_group/case_001_pre.h5  slices=4  lesion_slices=1
[DEMO] saved=1 skipped_patients=0
[DEMO] h5 keys: ['SliceThickness', 'b_value', 'dcm', 'label', 'meta']
[DEMO] dcm shape: (4, 192, 192), label shape: (4, 192, 192)
[DEMO] b_value: [1000 1000 1000 1000]
[DEMO] lesion voxel count per slice: [  0   0 576   0]
[DEMO] meta[0] (합성 익명 경로): <demo_tmp>/results_train_cleaned/demo_group/case_001/pre/0-dicom-0000.dcm

[DEMO] get_h5 파이프라인이 합성 데이터에서 정상 실행됨을 확인.
